# Hetionet 데이터 다운로드 및 서브그래프 추출

이 노트북은 Hetionet v1.0 생물의학 지식 그래프에서 **Gene-Pathway-Disease 서브그래프**를 추출하는 전체 과정을 다룹니다.

**파이프라인 전체 흐름:**
```
Hetionet v1.0                    TCGA-KIRC
(47K nodes, 2.25M edges)         (RNA-seq, ~600 samples)
        │                              │
   [Part 1] 구조 탐색            [Part 3] 다운로드 + TPM 통일
        │                              │
   [Part 2] 서브그래프 추출             │
        │                              │
        └──────── [Part 4] ────────────┘
                    │
              Gene ID 매칭
          (Entrez ↔ Ensembl)
                    │
              [Part 5] 최종 서브그래프
```

**이 노트북의 범위:** Part 1 (구조 탐색) + Part 2 (서브그래프 추출) + Part 3 (KIRC 다운로드)

Gene ID 매칭(Part 4)은 별도 단계에서 진행합니다.

---
## 0. 환경 설정

In [1]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import Counter
import os, time

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"pandas {pd.__version__}")
print(f"networkx {nx.__version__}")
print(f"numpy {np.__version__}")

pandas 3.0.3
networkx 3.6.1
numpy 2.4.6


---
## 1. Hetionet 데이터 다운로드

### Hetionet이란?

Hetionet v1.0은 29개 공개 데이터베이스를 통합한 **생물의학 이종 네트워크(heterogeneous biomedical network)**입니다.

- **47,031개 노드** (11가지 타입: Gene, Disease, Compound, Pathway 등)
- **2,250,197개 엣지** (24가지 관계 타입)

데이터는 두 파일로 배포됩니다:

| 파일 | 내용 | 크기 |
|------|------|------|
| `nodes.tsv` | 모든 노드 정보 (id, name, kind) | ~2MB |
| `edges.sif.gz` | 모든 엣지 정보 (source, metaedge, target) | ~12MB (압축) |

### 다운로드 시 주의사항

`edges.sif.gz`는 **Git LFS**로 관리됩니다. `raw.githubusercontent.com`으로 받으면 133바이트짜리 LFS 포인터만 받게 됩니다. 반드시 `media.githubusercontent.com` 경로를 사용해야 합니다.

In [2]:
# nodes.tsv 다운로드 (일반 GitHub raw URL)
# 이미 파일이 있으면 건너뜁니다.

nodes_path = os.path.join(DATA_DIR, "nodes.tsv")

if os.path.exists(nodes_path):
    print(f"이미 존재: {nodes_path} ({os.path.getsize(nodes_path)/1e6:.1f} MB)")
else:
    print("nodes.tsv 다운로드 중...")
    import urllib.request
    url = "https://raw.githubusercontent.com/hetio/hetionet/main/hetnet/tsv/hetionet-v1.0-nodes.tsv"
    urllib.request.urlretrieve(url, nodes_path)
    print(f"완료: {os.path.getsize(nodes_path)/1e6:.1f} MB")

nodes.tsv 다운로드 중...
완료: 2.5 MB


In [3]:
# edges.sif 다운로드
# ★ Git LFS 파일이므로 media.githubusercontent.com 사용

edges_path = os.path.join(DATA_DIR, "edges.sif")
edges_gz_path = edges_path + ".gz"

if os.path.exists(edges_path) and os.path.getsize(edges_path) > 1_000_000:
    print(f"이미 존재: {edges_path} ({os.path.getsize(edges_path)/1e6:.1f} MB)")
else:
    import urllib.request, gzip, shutil
    url = "https://media.githubusercontent.com/media/hetio/hetionet/main/hetnet/tsv/hetionet-v1.0-edges.sif.gz"
    print(f"edges.sif.gz 다운로드 중...")
    print(f"  URL: {url}")
    print(f"  (raw.githubusercontent.com이 아닌 media URL을 써야 LFS 실제 파일을 받음)")
    urllib.request.urlretrieve(url, edges_gz_path)
    print(f"  다운로드 완료: {os.path.getsize(edges_gz_path)/1e6:.1f} MB (압축)")

    # 압축 해제
    print("  압축 해제 중...")
    with gzip.open(edges_gz_path, 'rb') as f_in:
        with open(edges_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"  완료: {os.path.getsize(edges_path)/1e6:.1f} MB")
    os.remove(edges_gz_path)

edges.sif.gz 다운로드 중...
  URL: https://media.githubusercontent.com/media/hetio/hetionet/main/hetnet/tsv/hetionet-v1.0-edges.sif.gz
  (raw.githubusercontent.com이 아닌 media URL을 써야 LFS 실제 파일을 받음)
  다운로드 완료: 12.4 MB (압축)
  압축 해제 중...
  완료: 91.2 MB


---
## 2. 원본 데이터 로드 및 기본 구조 파악

### 파일 구조

**nodes.tsv** — 3개 컬럼:
- `id`: `타입::식별자` 형식 (예: `Gene::7157`)
- `name`: 사람이 읽을 수 있는 이름 (예: `TP53`)
- `kind`: 노드 타입 (예: `Gene`)

**edges.sif** — 3개 컬럼:
- `source`: 출발 노드 ID (예: `Gene::9021`)
- `metaedge`: 엣지 타입 약어 (예: `GpPW`)
- `target`: 도착 노드 ID (예: `Pathway::PC7_6941`)

In [4]:
# 데이터 로드
nodes = pd.read_csv(f"{DATA_DIR}/nodes.tsv", sep="\t")
edges = pd.read_csv(f"{DATA_DIR}/edges.sif", sep="\t")

# edges에 헤더 행이 데이터에 섞여 있을 수 있으므로 제거
edges = edges[edges["metaedge"] != "metaedge"]

print(f"전체 노드 수: {len(nodes):,}")
print(f"전체 엣지 수: {len(edges):,}")

전체 노드 수: 47,031
전체 엣지 수: 2,250,197


In [5]:
# nodes.tsv 구조 확인
print("=== nodes.tsv ===")
print(f"컬럼: {list(nodes.columns)}")
print()
nodes.head(10)

=== nodes.tsv ===
컬럼: ['id', 'name', 'kind']



,id,name,kind
0,Anatomy::UBERON:0000002,uterine cervix,Anatomy
1,Anatomy::UBERON:0000004,nose,Anatomy
2,Anatomy::UBERON:0000006,islet of Langerhans,Anatomy
3,Anatomy::UBERON:0000007,pituitary gland,Anatomy
4,Anatomy::UBERON:0000010,peripheral nervous system,Anatomy
5,Anatomy::UBERON:0000011,parasympathetic nervous system,Anatomy
6,Anatomy::UBERON:0000013,sympathetic nervous system,Anatomy
7,Anatomy::UBERON:0000020,sense organ,Anatomy
8,Anatomy::UBERON:0000026,appendage,Anatomy
9,Anatomy::UBERON:0000029,lymph node,Anatomy


In [6]:
# edges.sif 구조 확인
print("=== edges.sif ===")
print(f"컬럼: {list(edges.columns)}")
print()
edges.head(10)

=== edges.sif ===
컬럼: ['source', 'metaedge', 'target']



,source,metaedge,target
0,Gene::9021,GpBP,Biological Process::GO:0071357
1,Gene::51676,GpBP,Biological Process::GO:0098780
2,Gene::19,GpBP,Biological Process::GO:0055088
3,Gene::3176,GpBP,Biological Process::GO:0010243
4,Gene::3039,GpBP,Biological Process::GO:0006898
5,Gene::5962,GpBP,Biological Process::GO:0051346
6,Gene::841,GpBP,Biological Process::GO:0043207
7,Gene::6924,GpBP,Biological Process::GO:0006354
8,Gene::7407,GpBP,Biological Process::GO:0006417
9,Gene::9370,GpBP,Biological Process::GO:0030852


---
## 3. 노드 타입 분석 (11종)

Hetionet에는 11가지 노드 타입이 있습니다. 각 타입이 생물학적으로 무엇을 의미하는지, 그리고 어떤 **식별자 체계**를 쓰는지 파악하는 것이 중요합니다.

In [7]:
# 노드 타입별 개수
node_counts = nodes["kind"].value_counts()

print(f"{'타입':<25s} {'개수':>8s}")
print("-" * 35)
for kind, count in node_counts.items():
    print(f"{kind:<25s} {count:>8,}")

print(f"{'─'*35}")
print(f"{'합계':<25s} {node_counts.sum():>8,}")

타입                              개수
-----------------------------------
Gene                        20,945
Biological Process          11,381
Side Effect                  5,734
Molecular Function           2,884
Pathway                      1,822
Compound                     1,552
Cellular Component           1,391
Symptom                        438
Anatomy                        402
Pharmacologic Class            345
Disease                        137
───────────────────────────────────
합계                          47,031


### 우리가 사용할 노드 타입 4종

| 타입 | 개수 | 의미 | 식별자 체계 |
|------|------|------|-------------|
| **Gene** | 20,945 | 유전자 | Entrez Gene ID (정수) |
| **Biological Process** | 11,381 | 생물학적 과정 | GO ID |
| **Pathway** | 1,822 | 생물학적 경로 | PC7_xxx / WPxxx |
| **Disease** | 137 | 질병 | DOID |

나머지 7종(Compound, Side Effect, Molecular Function, Cellular Component, Symptom, Anatomy, Pharmacologic Class)은 약물 분석이나 해부학 연구에 필요한 것들로, 현재 Gene-Pathway-Disease 분석에서는 제외합니다.

---
## 4. 식별자(ID) 체계 분석

Hetionet의 `id` 컬럼은 **이름이 아니라 표준 식별자(ID)**를 사용합니다. 이것은 매우 중요한 사실인데, 나중에 TCGA-KIRC 데이터와 매칭할 때 이 ID 체계를 기준으로 매핑해야 하기 때문입니다.

### 4-1. Gene 노드의 ID 체계

In [8]:
# Gene 노드의 ID 형식 확인
# Gene::7157 에서 7157은 Entrez Gene ID (NCBI가 부여한 고유 정수 식별자)

genes = nodes[nodes["kind"] == "Gene"].copy()
genes["entrez_id"] = genes["id"].str.split("::").str[1].astype(int)

print(f"Gene 노드 수: {len(genes):,}")
print(f"\nID 형식: Gene::<Entrez_Gene_ID>")
print(f"\n예시:")
print(f"{'Hetionet ID':<20s} {'Entrez ID':>12s}   {'Gene Symbol'}")
print("-" * 50)
for _, row in genes.head(8).iterrows():
    print(f"{row['id']:<20s} {row['entrez_id']:>12d}   {row['name']}")

print(f"\n★ TCGA-KIRC는 Ensembl ID (예: ENSG00000141510)를 사용")
print(f"  → Entrez ↔ Ensembl 매핑이 반드시 필요 (step4에서 수행)")

Gene 노드 수: 20,945

ID 형식: Gene::<Entrez_Gene_ID>

예시:
Hetionet ID             Entrez ID   Gene Symbol
--------------------------------------------------
Gene::1                         1   A1BG
Gene::10                       10   NAT2
Gene::100                     100   ADA
Gene::1000                   1000   CDH2
Gene::10000                 10000   AKT3
Gene::100008586         100008586   GAGE12F
Gene::10001                 10001   MED6
Gene::10002                 10002   NR2E3

★ TCGA-KIRC는 Ensembl ID (예: ENSG00000141510)를 사용
  → Entrez ↔ Ensembl 매핑이 반드시 필요 (step4에서 수행)


### 4-2. Pathway 노드의 ID 체계

Pathway 노드에는 **두 가지 접두사 체계**가 혼재합니다:

- `PC7_xxxxx` — Pathway Commons 7의 내부 ID. Reactome, KEGG, BioCyc 등 여러 경로 DB를 통합한 것 (1,528개)
- `WPxxx_rNNNNN` — WikiPathways ID + revision 번호 (294개)

이름이 아닌 **ID 기반 매칭**이 안정적입니다 (이름은 DB마다 표기가 다를 수 있음).

In [9]:
# Pathway 노드의 ID 형식 확인
pathways = nodes[nodes["kind"] == "Pathway"].copy()
pathways["raw_id"] = pathways["id"].str.split("::").str[1]
pathways["db_prefix"] = pathways["raw_id"].str.extract(r"^([A-Za-z]+\d*)")

print(f"Pathway 노드 수: {len(pathways):,}")

# 접두사별 개수
prefix_counts = pathways["db_prefix"].value_counts()
print(f"\n접두사별 분류:")
for prefix, count in prefix_counts.items():
    if prefix == "PC7":
        db_name = "Pathway Commons 7 (Reactome/KEGG/BioCyc 등 통합)"
    elif prefix.startswith("WP"):
        db_name = "WikiPathways"
    else:
        db_name = "Unknown"
    print(f"  {prefix}: {count:,}개 — {db_name}")

# 예시 출력
print(f"\n예시:")
for _, row in pathways.head(5).iterrows():
    print(f"  {row['id']:<35s} → {row['name']}")

Pathway 노드 수: 1,822

접두사별 분류:
  PC7: 1,528개 — Pathway Commons 7 (Reactome/KEGG/BioCyc 등 통합)
  WP100: 1개 — WikiPathways
  WP106: 1개 — WikiPathways
  WP107: 1개 — WikiPathways
  WP111: 1개 — WikiPathways
  WP117: 1개 — WikiPathways
  WP127: 1개 — WikiPathways
  WP129: 1개 — WikiPathways
  WP12: 1개 — WikiPathways
  WP134: 1개 — WikiPathways
  WP136: 1개 — WikiPathways
  WP138: 1개 — WikiPathways
  WP1403: 1개 — WikiPathways
  WP1422: 1개 — WikiPathways
  WP1423: 1개 — WikiPathways
  WP1424: 1개 — WikiPathways
  WP1425: 1개 — WikiPathways
  WP1433: 1개 — WikiPathways
  WP1434: 1개 — WikiPathways
  WP143: 1개 — WikiPathways
  WP1449: 1개 — WikiPathways
  WP1455: 1개 — WikiPathways
  WP1471: 1개 — WikiPathways
  WP1528: 1개 — WikiPathways
  WP1530: 1개 — WikiPathways
  WP1531: 1개 — WikiPathways
  WP1533: 1개 — WikiPathways
  WP1539: 1개 — WikiPathways
  WP1544: 1개 — WikiPathways
  WP1545: 1개 — WikiPathways
  WP1559: 1개 — WikiPathways
  WP1584: 1개 — WikiPathways
  WP1589: 1개 — WikiPathways
  WP1591: 1개 — WikiPathwa

### 4-3. Disease 노드의 ID 체계

In [10]:
# Disease 노드 확인 — Disease Ontology ID (DOID) 사용
diseases = nodes[nodes["kind"] == "Disease"].copy()

print(f"Disease 노드 수: {len(diseases):,}")
print(f"\n신장(kidney) 관련 Disease:")

kidney_mask = diseases["name"].str.lower().str.contains("kidney|renal|clear cell")
kidney_diseases = diseases[kidney_mask]

for _, row in kidney_diseases.iterrows():
    print(f"  {row['id']:<25s} → {row['name']}")

print(f"\n★ TCGA-KIRC(Kidney Renal Clear Cell Carcinoma)와 직접 관련된 노드:")
print(f"  Disease::DOID:263 (kidney cancer)")

Disease 노드 수: 137

신장(kidney) 관련 Disease:
  Disease::DOID:263         → kidney cancer
  Disease::DOID:3953        → adrenal gland cancer
  Disease::DOID:784         → chronic kidney failure

★ TCGA-KIRC(Kidney Renal Clear Cell Carcinoma)와 직접 관련된 노드:
  Disease::DOID:263 (kidney cancer)


---
## 5. 엣지 타입(metaedge) 분석 (24종)

### 메타엣지 약어 읽는 법

Hetionet의 엣지 약어는 규칙적입니다:
- **앞글자** = source 노드 타입의 머리글자
- **뒷글자** = target 노드 타입의 머리글자
- **중간** = 관계를 나타내는 약어
- `>` = 방향성 표시

예시:
- `GpPW` = **G**ene **p**articipates **P**ath**W**ay
- `Gr>G` = **G**ene **r**egulates **G**ene (방향 있음)
- `DaG`  = **D**isease **a**ssociates **G**ene

In [11]:
# 전체 24종 엣지 타입 확인
METAEDGE_NAMES = {
    "GpBP": "Gene → participates → Biological Process",
    "AeG":  "Anatomy → expresses → Gene",
    "Gr>G": "Gene → regulates → Gene",
    "GiG":  "Gene → interacts → Gene",
    "CcSE": "Compound → causes → Side Effect",
    "AdG":  "Anatomy → downregulates → Gene",
    "AuG":  "Anatomy → upregulates → Gene",
    "GpMF": "Gene → participates → Molecular Function",
    "GpPW": "Gene → participates → Pathway",
    "GpCC": "Gene → participates → Cellular Component",
    "GcG":  "Gene → covaries → Gene",
    "CdG":  "Compound → downregulates → Gene",
    "CuG":  "Compound → upregulates → Gene",
    "DaG":  "Disease → associates → Gene",
    "CbG":  "Compound → binds → Gene",
    "DuG":  "Disease → upregulates → Gene",
    "DdG":  "Disease → downregulates → Gene",
    "CrC":  "Compound → resembles → Compound",
    "DlA":  "Disease → localizes → Anatomy",
    "DpS":  "Disease → presents → Symptom",
    "PCiC": "Pharmacologic Class → includes → Compound",
    "CtD":  "Compound → treats → Disease",
    "DrD":  "Disease → resembles → Disease",
    "CpD":  "Compound → palliates → Disease",
}

edge_counts = edges["metaedge"].value_counts()

print(f"{'약어':<6s} {'개수':>10s}   {'의미'}")
print("─" * 65)
for me, count in edge_counts.items():
    desc = METAEDGE_NAMES.get(me, "???")
    print(f"{me:<6s} {count:>10,}   {desc}")

print(f"{'─'*65}")
print(f"{'합계':<6s} {edge_counts.sum():>10,}")

약어             개수   의미
─────────────────────────────────────────────────────────────────
GpBP      559,504   Gene → participates → Biological Process
AeG       526,407   Anatomy → expresses → Gene
Gr>G      265,672   Gene → regulates → Gene
GiG       147,164   Gene → interacts → Gene
CcSE      138,944   Compound → causes → Side Effect
AdG       102,240   Anatomy → downregulates → Gene
AuG        97,848   Anatomy → upregulates → Gene
GpMF       97,222   Gene → participates → Molecular Function
GpPW       84,372   Gene → participates → Pathway
GpCC       73,566   Gene → participates → Cellular Component
GcG        61,690   Gene → covaries → Gene
CdG        21,102   Compound → downregulates → Gene
CuG        18,756   Compound → upregulates → Gene
DaG        12,623   Disease → associates → Gene
CbG        11,571   Compound → binds → Gene
DuG         7,731   Disease → upregulates → Gene
DdG         7,623   Disease → downregulates → Gene
CrC         6,486   Compound → resembles → Compound
Dl

---
## 6. 서브그래프 추출 — 설계

전체 47K 노드 / 2.25M 엣지 중에서 **Gene-Pathway-Disease 관계 분석**에 필요한 부분만 추출합니다.

### 추출할 노드 타입 (4종)

| 타입 | 이유 |
|------|------|
| Gene | 분석의 핵심 — TCGA 발현 데이터와 매칭 대상 |
| Pathway | Gene이 참여하는 생물학적 경로 |
| Disease | Gene과 연관된 질병 (KIRC 포함) |
| Biological Process | Gene의 기능적 역할 |

### 추출할 엣지 타입 (7종)

| 약어 | 의미 | 왜 필요한가 |
|------|------|------------|
| `GpPW` | Gene → Pathway | 유전자가 어떤 경로에 참여하는지 |
| `GpBP` | Gene → Biological Process | 유전자의 기능 분류 |
| `GiG` | Gene ↔ Gene (상호작용) | 유전자 간 물리적 상호작용 |
| `Gr>G` | Gene → Gene (조절) | 유전자 간 조절 관계 |
| `DaG` | Disease → Gene (연관) | 질병과 유전자의 연관성 |
| `DuG` | Disease → Gene (상향조절) | 질병 시 발현 증가 유전자 |
| `DdG` | Disease → Gene (하향조절) | 질병 시 발현 감소 유전자 |

### 구현 전략: pandas 선행 필터링

2.25M 엣지를 전부 NetworkX에 올린 후 필터하면 느립니다 (수 분 소요). 대신 **pandas의 벡터 연산으로 먼저 필터링**한 뒤 NetworkX에 넣으면 ~10초면 끝납니다.

```
[느린 방법]  전체 엣지 → NetworkX 로드 → 필터링  (2분+)
[빠른 방법]  pandas 필터링 → 필요한 것만 NetworkX  (~10초)
```

In [12]:
# ── 추출 기준 정의 ──

KEEP_NODE_TYPES = {"Gene", "Pathway", "Disease", "Biological Process"}

KEEP_EDGE_TYPES = {
    "GpPW",   # Gene → participates → Pathway
    "GiG",    # Gene → interacts → Gene
    "Gr>G",   # Gene → regulates → Gene
    "DaG",    # Disease → associates → Gene
    "DuG",    # Disease → upregulates → Gene
    "DdG",    # Disease → downregulates → Gene
    "GpBP",   # Gene → participates → Biological Process
}

print(f"유지할 노드 타입 ({len(KEEP_NODE_TYPES)}종): {KEEP_NODE_TYPES}")
print(f"유지할 엣지 타입 ({len(KEEP_EDGE_TYPES)}종): {KEEP_EDGE_TYPES}")

유지할 노드 타입 (4종): {'Gene', 'Disease', 'Pathway', 'Biological Process'}
유지할 엣지 타입 (7종): {'Gr>G', 'GpPW', 'DuG', 'DaG', 'GiG', 'DdG', 'GpBP'}


### 6-1. 노드 필터링 + 엣지 pandas 필터링

In [13]:
t0 = time.time()

# ── Step A: 유지할 노드 ID 집합 구성 ──
# nodes DataFrame에서 4종 타입에 해당하는 노드의 id만 set으로 모은다.
# set 자료구조이므로 나중에 isin() 호출 시 O(1) lookup이 가능.

keep_nodes_df = nodes[nodes["kind"].isin(KEEP_NODE_TYPES)]
keep_node_ids = set(keep_nodes_df["id"])

print(f"유지할 노드 수: {len(keep_node_ids):,}")
for kind, count in keep_nodes_df["kind"].value_counts().items():
    print(f"  {kind:<25s} {count:>6,}")

# ── Step B: 엣지 벡터화 필터링 ──
# 2,250,197개 엣지에 대해 3가지 조건을 한번에 적용.
# pandas의 isin()은 내부적으로 해시 기반이라 225만 행도 수 초면 끝남.

mask = (
    edges["metaedge"].isin(KEEP_EDGE_TYPES) &   # 7종 엣지 타입만
    edges["source"].isin(keep_node_ids) &         # source가 4종 노드에 포함
    edges["target"].isin(keep_node_ids)            # target도 4종 노드에 포함
)

filtered_edges = edges[mask]

print(f"\n전체 엣지: {len(edges):,}")
print(f"필터 후 엣지: {len(filtered_edges):,}")
print(f"필터링 소요시간: {time.time()-t0:.1f}초")

유지할 노드 수: 34,285
  Gene                      20,945
  Biological Process        11,381
  Pathway                    1,822
  Disease                      137

전체 엣지: 2,250,197
필터 후 엣지: 1,084,689
필터링 소요시간: 0.4초


In [14]:
# 필터된 엣지의 타입별 분포 확인
print("필터된 엣지 타입별 개수:")
print()
for me, count in filtered_edges["metaedge"].value_counts().items():
    desc = METAEDGE_NAMES.get(me, "")
    print(f"  {me:<6s} {count:>8,}   {desc}")

필터된 엣지 타입별 개수:

  GpBP    559,504   Gene → participates → Biological Process
  Gr>G    265,672   Gene → regulates → Gene
  GiG     147,164   Gene → interacts → Gene
  GpPW     84,372   Gene → participates → Pathway
  DaG      12,623   Disease → associates → Gene
  DuG       7,731   Disease → upregulates → Gene
  DdG       7,623   Disease → downregulates → Gene


### 6-2. NetworkX 그래프 구축

`nx.MultiDiGraph`를 사용하는 이유:
- **Multi**: 같은 노드 쌍 사이에 여러 종류의 엣지가 존재 가능 (예: Gene A → Gene B 사이에 `GiG`와 `Gr>G` 동시 존재)
- **Di**: 방향이 있는 그래프 (`DaG`: Disease → Gene 방향)

In [15]:
t0 = time.time()

# ── 노드 정보 딕셔너리 (빠른 lookup용) ──
node_info = {}
for row in nodes.itertuples(index=False):
    node_info[row.id] = {"name": row.name, "kind": row.kind}

# ── MultiDiGraph 생성 ──
G = nx.MultiDiGraph()

# 노드 추가 (name, kind 속성 포함)
for nid in keep_node_ids:
    info = node_info[nid]
    G.add_node(nid, name=info["name"], kind=info["kind"])

# 엣지 추가 (metaedge 속성 포함)
for row in filtered_edges.itertuples(index=False):
    G.add_edge(row.source, row.target, metaedge=row.metaedge)

print(f"그래프 구축 완료 ({time.time()-t0:.1f}초)")
print(f"  노드: {G.number_of_nodes():,}")
print(f"  엣지: {G.number_of_edges():,}")

그래프 구축 완료 (4.7초)
  노드: 34,285
  엣지: 1,084,689


### 6-3. 고립 노드 제거

엣지 타입 필터링 후, 어떤 엣지와도 연결되지 않은 노드가 생길 수 있습니다. 예를 들어 Disease 137개 중 일부는 우리가 선택한 7종 엣지와 아무 연결이 없을 수 있습니다. 이런 고립 노드를 제거합니다.

In [16]:
# 고립 노드 = degree가 0인 노드
isolates = list(nx.isolates(G))

print(f"고립 노드 수: {len(isolates):,}")

# 고립 노드의 타입 분포
if isolates:
    iso_types = Counter(G.nodes[n]["kind"] for n in isolates)
    print("  타입별:")
    for kind, count in iso_types.most_common():
        print(f"    {kind}: {count}개")

# 제거
G.remove_nodes_from(isolates)

print(f"\n최종 서브그래프:")
print(f"  노드: {G.number_of_nodes():,}")
print(f"  엣지: {G.number_of_edges():,}")

고립 노드 수: 2,674
  타입별:
    Gene: 2671개
    Disease: 3개

최종 서브그래프:
  노드: 31,611
  엣지: 1,084,689


---
## 7. 서브그래프 구성 분석

In [17]:
# ── 노드 타입별 개수 ──
sub_node_types = Counter(d["kind"] for _, d in G.nodes(data=True))

print("[노드 타입별 개수]")
print(f"{'타입':<25s} {'개수':>8s}")
print("─" * 35)
for kind, cnt in sub_node_types.most_common():
    print(f"{kind:<25s} {cnt:>8,}")

[노드 타입별 개수]
타입                              개수
───────────────────────────────────
Gene                        18,274
Biological Process          11,381
Pathway                      1,822
Disease                        134


In [18]:
# ── 엣지 타입별 개수 ──
sub_edge_types = Counter(d["metaedge"] for _, _, d in G.edges(data=True))

EDGE_DESC = {
    "GpBP": "Gene → Biological Process",
    "Gr>G": "Gene → regulates → Gene",
    "GiG":  "Gene ↔ Gene (상호작용)",
    "GpPW": "Gene → Pathway",
    "DaG":  "Disease → Gene (연관)",
    "DuG":  "Disease → Gene (상향조절)",
    "DdG":  "Disease → Gene (하향조절)",
}

print("[엣지 타입별 개수]")
print(f"{'약어':<6s} {'개수':>10s}   {'의미'}")
print("─" * 55)
for me, cnt in sub_edge_types.most_common():
    print(f"{me:<6s} {cnt:>10,}   {EDGE_DESC.get(me, '')}")

[엣지 타입별 개수]
약어             개수   의미
───────────────────────────────────────────────────────
GpBP      559,504   Gene → Biological Process
Gr>G      265,672   Gene → regulates → Gene
GiG       147,164   Gene ↔ Gene (상호작용)
GpPW       84,372   Gene → Pathway
DaG        12,623   Disease → Gene (연관)
DuG         7,731   Disease → Gene (상향조절)
DdG         7,623   Disease → Gene (하향조절)


---
## 8. Gene-Pathway 연결 통계

Pathway당 몇 개의 Gene이 참여하는지 분포를 확인합니다.

In [19]:
# 각 Pathway에 연결된 Gene 수 계산
# GpPW 엣지 방향: Gene → Pathway (Gene이 source, Pathway가 target)
# 따라서 Pathway의 in_edges 중 metaedge="GpPW"인 것의 source가 Gene

pathway_nodes = [n for n, d in G.nodes(data=True) if d["kind"] == "Pathway"]

pw_gene_counts = {}
for pw in pathway_nodes:
    gene_count = sum(
        1 for u, v, d in G.in_edges(pw, data=True)
        if d.get("metaedge") == "GpPW"
    )
    pw_gene_counts[pw] = gene_count

pw_sizes = sorted(pw_gene_counts.values(), reverse=True)

print(f"Pathway 수: {len(pathway_nodes):,}")
print(f"Pathway당 Gene 수:")
print(f"  최소: {min(pw_sizes)}")
print(f"  최대: {max(pw_sizes)}")
print(f"  중앙값: {pw_sizes[len(pw_sizes)//2]}")
print(f"  평균: {sum(pw_sizes)/len(pw_sizes):.1f}")

Pathway 수: 1,822
Pathway당 Gene 수:
  최소: 2
  최대: 1956
  중앙값: 23
  평균: 46.3


In [20]:
# 상위 15개 Pathway (Gene 수 기준)
top15 = sorted(pw_gene_counts.items(), key=lambda x: x[1], reverse=True)[:15]

print("상위 15개 Pathway (Gene 수 기준):")
print(f"{'#':<4s} {'Gene수':>6s}   {'ID':<35s} {'이름'}")
print("─" * 80)
for i, (pw_id, cnt) in enumerate(top15, 1):
    pw_name = G.nodes[pw_id]["name"]
    print(f"{i:<4d} {cnt:>6d}   {pw_id:<35s} {pw_name}")

상위 15개 Pathway (Gene 수 기준):
#     Gene수   ID                                  이름
────────────────────────────────────────────────────────────────────────────────
1      1956   Pathway::PC7_7439                   Signaling Pathways
2      1564   Pathway::PC7_5322                   Metabolism
3      1253   Pathway::PC7_3278                   Disease
4      1142   Pathway::PC7_4112                   Gene Expression
5      1134   Pathway::PC7_4688                   Immune System
6      1013   Pathway::PC7_7457                   Signaling by GPCR
7       898   Pathway::PC7_4043                   GPCR downstream signaling
8       683   Pathway::PC7_5330                   Metabolism of proteins
9       607   Pathway::PC7_1563                   Adaptive Immune System
10      606   Pathway::PC7_8339                   Transmembrane transport of small molecules
11      577   Pathway::PC7_4741                   Innate Immune System
12      569   Pathway::PC7_5326                   Metabolism of li

---
## 9. Disease-Gene 연결 분석 (KIRC 관련)

TCGA-KIRC(신장 투명세포암)와 관련된 Disease 노드가 서브그래프에 포함되어 있는지 확인하고, 연결된 Gene 수를 봅니다.

In [21]:
# 신장/신암 관련 Disease 검색
disease_nodes = [n for n, d in G.nodes(data=True) if d["kind"] == "Disease"]
print(f"서브그래프 내 Disease 수: {len(disease_nodes)}")

# 신장 관련 키워드로 검색
keywords = ["kidney", "renal", "clear cell"]
print(f"\n신장 관련 Disease 노드:")
print("─" * 70)

for dn in disease_nodes:
    name = G.nodes[dn]["name"].lower()
    if any(kw in name for kw in keywords):
        # 각 엣지 타입별 연결 Gene 수
        out_edges = list(G.out_edges(dn, data=True))
        edge_by_type = Counter(d["metaedge"] for _, _, d in out_edges)
        total_genes = sum(edge_by_type.values())

        print(f"\n  {G.nodes[dn]['name']} ({dn})")
        print(f"  총 연결 Gene: {total_genes}개")
        for me, cnt in edge_by_type.most_common():
            print(f"    {me} ({EDGE_DESC.get(me, '')}): {cnt}개")

서브그래프 내 Disease 수: 134

신장 관련 Disease 노드:
──────────────────────────────────────────────────────────────────────

  chronic kidney failure (Disease::DOID:784)
  총 연결 Gene: 132개
    DaG (Disease → Gene (연관)): 132개

  kidney cancer (Disease::DOID:263)
  총 연결 Gene: 713개
    DdG (Disease → Gene (하향조절)): 250개
    DuG (Disease → Gene (상향조절)): 250개
    DaG (Disease → Gene (연관)): 213개

  adrenal gland cancer (Disease::DOID:3953)
  총 연결 Gene: 38개
    DaG (Disease → Gene (연관)): 38개


---
## 10. TCGA-KIRC 발현 데이터 다운로드

파이프라인 Part 3에 해당합니다. UCSC Xena의 **GDC hub**에서 TCGA-KIRC 코호트의
RNA-seq 발현값과 임상 정보를 받습니다.

| 파일 | 내용 | 크기 |
|------|------|------|
| `TCGA-KIRC.star_tpm.tsv.gz` | STAR 정렬 후 TPM, `log2(TPM + 1)` 변환 | ~178 MB |
| `TCGA-KIRC.clinical.tsv.gz` | 임상 정보 | ~150 KB |

### 식별자 체계

발현 행렬의 행 인덱스는 **버전 접미사가 붙은 Ensembl ID**입니다 (예: `ENSG00000141510.17`).
Hetionet Gene 노드는 Entrez ID를 쓰므로 두 체계를 잇는 매핑이 필요하지만,
그 작업은 **Part 4에서 별도로** 진행합니다. 여기서는 원본 형태 그대로 받아둡니다.

### 다운로드 시 주의사항

Xena 공식 호스트(`tcga.xenahubs.net` / `gdc.xenahubs.net`)가 막힌 망에서는
동일 데이터를 서빙하는 S3 버킷(`gdc-hub.s3.us-east-1.amazonaws.com`)으로 폴백합니다.

In [ ]:
# ── TCGA-KIRC 다운로드 ──
# 두 엔드포인트를 순서대로 시도한다.
#   1) gdc.xenahubs.net             — Xena 공식 호스트
#   2) gdc-hub.s3...amazonaws.com   — 동일 데이터의 S3 버킷 (1번이 막힌 망에서 사용)

import urllib.request, urllib.error

XENA_GDC_HOST = "https://gdc.xenahubs.net/download"
XENA_GDC_S3   = "https://gdc-hub.s3.us-east-1.amazonaws.com/download"

KIRC_FILES = [
    "TCGA-KIRC.star_tpm.tsv.gz",    # log2(TPM + 1), Ensembl ID
    "TCGA-KIRC.clinical.tsv.gz",    # 임상 정보
]

def fetch_xena(fname, min_bytes=100_000):
    """Xena에서 fname을 DATA_DIR로 받는다. 이미 받았으면 건너뛴다."""
    dest = os.path.join(DATA_DIR, fname)

    if os.path.exists(dest) and os.path.getsize(dest) > min_bytes:
        print(f"  이미 존재: {fname} ({os.path.getsize(dest)/1e6:.1f} MB)")
        return dest

    for base in (XENA_GDC_HOST, XENA_GDC_S3):
        url = f"{base}/{fname}"
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"  완료: {fname} ({os.path.getsize(dest)/1e6:.1f} MB)")
            print(f"    <- {base}")
            return dest
        except (urllib.error.URLError, OSError) as e:
            print(f"  실패 ({type(e).__name__}): {url}")

    raise RuntimeError(f"모든 엔드포인트에서 다운로드 실패: {fname}")

t0 = time.time()
for fname in KIRC_FILES:
    print(f"[{fname}]")
    fetch_xena(fname)
print(f"\n소요시간: {time.time()-t0:.1f}초")

In [ ]:
# ── 발현 행렬 로드 및 구조 확인 ──
kirc_expr = pd.read_csv(
    os.path.join(DATA_DIR, "TCGA-KIRC.star_tpm.tsv.gz"),
    sep="\t", index_col=0,
)

print(f"발현 행렬: {kirc_expr.shape[0]:,} genes x {kirc_expr.shape[1]:,} samples")
print(f"값 범위: {kirc_expr.to_numpy().min():.2f} ~ {kirc_expr.to_numpy().max():.2f}  (log2(TPM+1))")
print(f"\n행 인덱스 = 버전 접미사가 붙은 Ensembl ID:")
print(f"  {list(kirc_expr.index[:3])}")
print(f"\n열 = TCGA 바코드:")
print(f"  {list(kirc_expr.columns[:3])}")

kirc_expr.iloc[:5, :4]

In [ ]:
# ── 시료 종류 분포 ──
# TCGA 바코드의 4번째 필드가 시료 종류 코드 (01=원발암, 11=정상조직)
# GDC 바코드는 뒤에 vial 문자가 붙으므로(예: 01A) 앞 두 자리만 본다.

SAMPLE_TYPE = {
    "01": "Primary Tumor",
    "05": "Additional - New Primary",
    "06": "Metastatic",
    "11": "Solid Tissue Normal",
}

sample_codes = kirc_expr.columns.str.split("-").str[3].str[:2]

print("[시료 종류]")
for code_, n in sample_codes.value_counts().items():
    print(f"  {code_}  {SAMPLE_TYPE.get(code_, '?'):<26s} {n:>4,}")
print(f"{'─'*40}")
print(f"  {'합계':<30s} {len(sample_codes):>4,}")

# 임상 정보
kirc_clin = pd.read_csv(
    os.path.join(DATA_DIR, "TCGA-KIRC.clinical.tsv.gz"),
    sep="\t", index_col=0, low_memory=False,
)
print(f"\n임상 정보: {kirc_clin.shape[0]:,} samples x {kirc_clin.shape[1]:,} fields")

---
## 11. 결과 저장

In [23]:
# ── 서브그래프 노드 저장 ──
sub_nodes_df = pd.DataFrame([
    {"id": n, "name": d["name"], "kind": d["kind"]}
    for n, d in G.nodes(data=True)
])
sub_nodes_df.to_csv(f"{DATA_DIR}/subgraph_nodes.tsv", sep="\t", index=False)

# ── 서브그래프 엣지 저장 ──
sub_edges_df = pd.DataFrame([
    {"source": u, "metaedge": d["metaedge"], "target": v}
    for u, v, d in G.edges(data=True)
])
sub_edges_df.to_csv(f"{DATA_DIR}/subgraph_edges.tsv", sep="\t", index=False)

print("저장 완료:")
print(f"  subgraph_nodes.tsv  : {len(sub_nodes_df):,} rows")
print(f"  subgraph_edges.tsv  : {len(sub_edges_df):,} rows")

저장 완료:
  subgraph_nodes.tsv  : 31,611 rows
  subgraph_edges.tsv  : 1,084,689 rows
  gene_id_mapping.tsv : 18,274 rows


In [24]:
# ── 저장된 파일 확인 ──
print("data/ 폴더 파일 목록:")
for f in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, f)
    size = os.path.getsize(fpath)
    if size > 1e6:
        print(f"  {f:<40s} {size/1e6:.1f} MB")
    else:
        print(f"  {f:<40s} {size/1e3:.1f} KB")

data/ 폴더 파일 목록:
  edges.sif                                91.2 MB
  gene_id_mapping.tsv                      441.1 KB
  nodes.tsv                                2.5 MB
  subgraph_edges.tsv                       41.7 MB
  subgraph_nodes.tsv                       1.6 MB


---
## 12. 최종 요약

### 데이터 흐름 정리

```
Hetionet v1.0 원본
  47,031 노드 (11종)  /  2,250,197 엣지 (24종)
        │
   [노드 필터] 4종만 유지: Gene, Pathway, Disease, BP
   [엣지 필터] 7종만 유지: GpPW, GiG, Gr>G, DaG, DuG, DdG, GpBP
   [고립 노드 제거]
        │
        ▼
서브그래프
  ~31,600 노드  /  ~1,084,000 엣지
  Gene ~18,200 / BP ~11,300 / Pathway 1,822 / Disease ~134
```

### 출력 파일

| 파일 | 내용 | 다음 단계에서의 용도 |
|------|------|---------------------|
| `subgraph_nodes.tsv` | 서브그래프 노드 목록 | step5에서 최종 그래프 구축 시 사용 |
| `subgraph_edges.tsv` | 서브그래프 엣지 목록 | step5에서 최종 그래프 구축 시 사용 |
| `TCGA-KIRC.star_tpm.tsv.gz` | KIRC 발현값 `log2(TPM+1)` | **step4에서 Gene ID 매칭 대상** |
| `TCGA-KIRC.clinical.tsv.gz` | KIRC 임상 정보 | 하위 분석용 |

### 핵심 발견

1. **Gene 노드는 Entrez Gene ID(정수)** 사용 → TCGA-KIRC는 Ensembl ID → 매핑 필수 (step4)
2. **Pathway 노드는 PC7(1,528개) + WP(294개)** 두 체계 혼재 → ID 기반 매칭이 안정적
3. **kidney cancer (DOID:263)**가 서브그래프에 포함, ~713개 Gene과 연결
4. pandas 선행 필터링으로 2.25M → ~1.08M 엣지 필터가 ~수 초에 완료

### 다음 단계

- **step4**: Ensembl → Entrez 매핑으로 Gene 교집합 산출 (로컬 실행, mygene 필요)
- **step5**: 교집합 Gene만 남긴 최종 서브그래프 + GraphML 저장